# SIMULATED ward graph (not real boundaries)

This notebook builds a SIMULATED spatial network of 198 wards for Bengaluru's HydroGrid water model.
Coordinates, demand, stress metrics, and topology are simulated for demonstration purposes.

In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
import torch
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt

# Fix random seed for reproducibility (SIMULATED data)
np.random.seed(42)

In [2]:
# 1. Create 198 wards
num_wards = 198
ward_ids = np.arange(num_wards)
names = [f'Ward_{i+1:03d}' for i in range(num_wards)]

# 2. Simulated coordinates in a 20 x 20 km square
x = np.random.uniform(0, 20, num_wards)
y = np.random.uniform(0, 20, num_wards)
dist_center = np.sqrt((x - 10)**2 + (y - 10)**2)

# Assign zones based on distance to center (10, 10)
zone = np.where(dist_center < 5, 'core', np.where(dist_center < 8, 'mid', 'periphery'))

pd.Series(zone).value_counts()

zone
core          39
mid           52
periphery    107
Name: count, dtype: int64

In [3]:
# 3. base_mld (lognormal, scaled so sum is exactly 2632, periphery higher)
periphery_bias = 1.0 + 0.05 * dist_center
raw_mld = np.random.lognormal(mean=2.0, sigma=0.4, size=num_wards) * periphery_bias
base_mld = (raw_mld / raw_mld.sum()) * 2632.0

# 4. tanker_dependence (0.05 core to 0.60 far periphery + noise)
max_dist = dist_center.max()
tanker_noise = np.random.normal(0, 0.02, size=num_wards)
tanker_dependence = np.clip(0.05 + 0.55 * (dist_center / max_dist) + tanker_noise, 0.05, 0.60)

# 5. groundwater_stress (0 to 1, growing with dist_center + noise)
gw_noise = np.random.normal(0, 0.05, size=num_wards)
groundwater_stress = np.clip((dist_center / max_dist) + gw_noise, 0.0, 1.0)

# Assemble wards DataFrame
wards = pd.DataFrame({
    'ward_id': ward_ids,
    'name': names,
    'x': x,
    'y': y,
    'dist_center': dist_center,
    'zone': zone,
    'base_mld': base_mld,
    'tanker_dependence': tanker_dependence,
    'groundwater_stress': groundwater_stress
})

wards.head().round(2)

,ward_id,name,x,y,dist_center,zone,base_mld,tanker_dependence,groundwater_stress
0,0,Ward_001,7.49,0.12,10.19,periphery,18.06,0.47,0.86
1,1,Ward_002,19.01,19.39,13.01,periphery,10.36,0.57,0.96
2,2,Ward_003,14.64,16.03,7.61,mid,17.03,0.37,0.60
3,3,Ward_004,11.97,13.82,4.29,core,23.23,0.20,0.36
4,4,Ward_005,3.12,4.24,8.98,periphery,10.87,0.43,0.74


In [4]:
# 6. Build graph: connect each ward to its 4 nearest neighbours
coords = np.column_stack([x, y])
nbrs = NearestNeighbors(n_neighbors=5).fit(coords)  # 5 neighbors includes self
distances, indices = nbrs.kneighbors(coords)

# Collect undirected edges
edge_dict = {}
for i in range(num_wards):
    for j in range(1, 5):
        neighbor = indices[i, j]
        d = distances[i, j]
        u, v = sorted([i, neighbor])
        edge_dict[(u, v)] = d

# Form bi-directional (undirected) edge list and compute weights
edge_list = []
raw_weights = []
for (u, v), d in edge_dict.items():
    w = 1.0 / (d + 0.5)
    # Add (u, v)
    edge_list.append((u, v))
    raw_weights.append(w)
    # Add (v, u)
    edge_list.append((v, u))
    raw_weights.append(w)

# Normalise weights to range 0 to 1
raw_weights = np.array(raw_weights)
norm_weights = (raw_weights - raw_weights.min()) / (raw_weights.max() - raw_weights.min())

src = [e[0] for e in edge_list]
dst = [e[1] for e in edge_list]

edges_df = pd.DataFrame({
    'src': src,
    'dst': dst,
    'weight': norm_weights
})

print(f'Constructed {len(edges_df)} directed edges ({len(edge_dict)} undirected connections).')

Constructed 960 directed edges (480 undirected connections).


In [5]:
# 7. Create torch tensors for graph representation
edge_index = torch.tensor([src, dst], dtype=torch.long)
edge_weight = torch.tensor(norm_weights, dtype=torch.float32)

print('edge_index shape:', edge_index.shape)
print('edge_weight shape:', edge_weight.shape)

edge_index shape: torch.Size([2, 960])
edge_weight shape: torch.Size([960])


In [6]:
# 8. Save ward table and edges table to CSV
os.makedirs('data', exist_ok=True)
wards.to_csv('data/wards.csv', index=False)
edges_df.to_csv('data/edges.csv', index=False)

print(f'Saved data/wards.csv ({len(wards)} rows)')
print(f'Saved data/edges.csv ({len(edges_df)} rows)')

Saved data/wards.csv (198 rows)
Saved data/edges.csv (960 rows)


In [7]:
# 9. Plot the SIMULATED ward graph
plt.figure(figsize=(9, 9))

# Draw edges as thin grey lines
for (u, v) in edge_dict.keys():
    plt.plot(
        [wards.loc[u, 'x'], wards.loc[v, 'x']],
        [wards.loc[u, 'y'], wards.loc[v, 'y']],
        color='grey',
        alpha=0.35,
        linewidth=0.8,
        zorder=1
    )

# Scatter wards colored by zone
colors = {'core': 'royalblue', 'mid': 'darkorange', 'periphery': 'forestgreen'}
for z in ['core', 'mid', 'periphery']:
    subset = wards[wards['zone'] == z]
    plt.scatter(
        subset['x'],
        subset['y'],
        c=colors[z],
        label=f'{z.capitalize()} ({len(subset)} wards)',
        s=50,
        zorder=2,
        edgecolors='black',
        linewidth=0.5
    )

plt.title('SIMULATED ward graph', fontsize=14, fontweight='bold')
plt.xlabel('X coordinate (km)')
plt.ylabel('Y coordinate (km)')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [8]:
# 10. Print validation checks
adj = coo_matrix((norm_weights, (src, dst)), shape=(num_wards, num_wards))
n_components, _ = connected_components(csgraph=adj, directed=False)

print('=== CHECKS ===')
print('wards.shape:', wards.shape)
print("wards['base_mld'].sum():", wards['base_mld'].sum())
print('Number of edges:', len(edges_df))
print('Number of connected components:', n_components)

=== CHECKS ===
wards.shape: (198, 9)
wards['base_mld'].sum(): 2632.0
Number of edges: 960
Number of connected components: 1
